# Medical Insurance Cost Prediction using Multiple Linear Regression
**Author:** Ishika  
**Dataset:** Medical Insurance Dataset (Kaggle: `mirichoi0218/insurance`)  
**Algorithm:** Multiple Linear Regression (MLR)  

---
### Project Overview & Objectives
The objective of this project is to analyze individual health, demographic, and lifestyle factors to predict annual medical insurance costs (`charges`). By employing Multiple Linear Regression, this study uncovers key drivers influencing insurance charges, evaluates model accuracy, and establishes an automated pipeline for healthcare cost estimation.

## 1. Environment Setup & Library Imports
Import essential Python libraries for data wrangling, visualization, statistical modeling, and performance evaluation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Configure plot aesthetics
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

print("Libraries imported successfully!")

## 2. Dataset Loading & Exploratory Data Inspection
Load the raw `insurance.csv` dataset and perform initial structural and statistical assessments.

In [ ]:
# Load raw insurance dataset
df_raw = pd.read_csv('insurance.csv')
print(f"Dataset Shape: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns\n")
df_raw.head()

In [ ]:
# Inspect schema, missing entries, and data types
print("--- Dataset Information ---")
df_raw.info()
print("\n--- Missing Values Count ---")
print(df_raw.isnull().sum())

In [ ]:
# Statistical Summary of Numerical Attributes
df_raw.describe().T

## 3. Exploratory Data Analysis (EDA) & Visualizations
Explore univariate distributions and bivariate relationships between demographic/lifestyle features and medical charges.

In [ ]:
# Target Variable Distribution (charges)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df_raw['charges'], kde=True, color='#2b5c8f', ax=ax[0])
ax[0].set_title('Distribution of Medical Charges', fontsize=14, fontweight='bold')
ax[0].set_xlabel('Charges ($)')
ax[0].set_ylabel('Frequency')

sns.boxplot(x=df_raw['charges'], color='#4a90e2', ax=ax[1])
ax[1].set_title('Boxplot of Medical Charges (Outliers Check)', fontsize=14, fontweight='bold')
ax[1].set_xlabel('Charges ($)')
plt.tight_layout()
plt.show()

In [ ]:
# Impact of Smoking on Medical Charges
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(x='smoker', y='charges', data=df_raw, palette=['#3498db', '#e74c3c'], ax=ax[0])
ax[0].set_title('Medical Charges by Smoking Status (Boxplot)', fontsize=13, fontweight='bold')
ax[0].set_xlabel('Smoker (yes / no)')
ax[0].set_ylabel('Charges ($)')

sns.violinplot(x='smoker', y='charges', data=df_raw, palette=['#3498db', '#e74c3c'], ax=ax[1])
ax[1].set_title('Distribution of Charges by Smoking Status (Violinplot)', fontsize=13, fontweight='bold')
ax[1].set_xlabel('Smoker (yes / no)')
ax[1].set_ylabel('Charges ($)')
plt.tight_layout()
plt.show()

In [ ]:
# BMI vs Charges colored by Smoking Status
plt.figure(figsize=(10, 6))
sns.scatterplot(x='bmi', y='charges', hue='smoker', palette=['#2ecc71', '#e74c3c'], data=df_raw, alpha=0.8, s=60)
plt.axvline(30, color='gray', linestyle='--', label='Obesity Threshold (BMI = 30)')
plt.title('Relationship between BMI and Medical Charges by Smoking Status', fontsize=14, fontweight='bold')
plt.xlabel('Body Mass Index (BMI)')
plt.ylabel('Charges ($)')
plt.legend(title='Smoker')
plt.show()

In [ ]:
# Age vs Charges colored by Smoking Status
plt.figure(figsize=(10, 6))
sns.scatterplot(x='age', y='charges', hue='smoker', palette=['#3498db', '#e67e22'], data=df_raw, alpha=0.8, s=60)
plt.title('Relationship between Age and Medical Charges', fontsize=14, fontweight='bold')
plt.xlabel('Age (Years)')
plt.ylabel('Charges ($)')
plt.legend(title='Smoker')
plt.show()

In [ ]:
# Charges by Region and Children
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(x='region', y='charges', data=df_raw, palette='Blues_d', ax=ax[0], capsize=0.1)
ax[0].set_title('Average Charges by US Geographic Region', fontsize=13, fontweight='bold')
ax[0].set_xlabel('Region')
ax[0].set_ylabel('Mean Charges ($)')

sns.barplot(x='children', y='charges', data=df_raw, palette='Purples_d', ax=ax[1], capsize=0.1)
ax[1].set_title('Average Charges by Number of Dependents (Children)', fontsize=13, fontweight='bold')
ax[1].set_xlabel('Number of Children')
ax[1].set_ylabel('Mean Charges ($)')
plt.tight_layout()
plt.show()

## 4. Data Preprocessing & Categorical Feature Encoding
Transform categorical variables into numerical representations:
- Binary Mapping: `sex` (`male`: 0, `female`: 1), `smoker` (`yes`: 1, `no`: 0)
- One-Hot Encoding: `region` (`northeast`, `northwest`, `southeast`, `southwest`) using dummy variables with `drop_first=True` to avoid multicollinearity (the dummy variable trap).
- Save cleaned dataset to `insurance_cleaned.csv`.

In [ ]:
# Create a working copy for preprocessing
df_cleaned = df_raw.copy()

# Binary mapping
df_cleaned['sex'] = df_cleaned['sex'].map({'male': 0, 'female': 1})
df_cleaned['smoker'] = df_cleaned['smoker'].map({'yes': 1, 'no': 0})

# One-hot encoding for region
df_cleaned = pd.get_dummies(df_cleaned, columns=['region'], drop_first=True, dtype=int)

# Persist the cleaned dataset
df_cleaned.to_csv('insurance_cleaned.csv', index=False)
print("Cleaned dataset saved successfully to 'insurance_cleaned.csv'!")
df_cleaned.head()

In [ ]:
# Correlation Matrix & Heatmap
plt.figure(figsize=(10, 8))
corr_matrix = df_cleaned.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5)
plt.title('Correlation Matrix of Processed Features', fontsize=14, fontweight='bold')
plt.show()

## 5. Feature Selection & Train-Test Splitting
Separate the feature matrix ($X$) from the target vector ($y$) and split into an 80% training set and a 20% test set.

In [ ]:
# Define Feature Matrix (X) and Target Vector (y)
X = df_cleaned.drop('charges', axis=1)
y = df_cleaned['charges']

# 80/20 Train-Test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print(f"Training instances: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Testing instances:  {X_test.shape[0]} samples, {X_test.shape[1]} features")

## 6. Multiple Linear Regression Model Training
Train the Multiple Linear Regression model on `X_train` and `y_train` using Ordinary Least Squares (OLS).

In [ ]:
# Instantiate and train the model
mlr_model = LinearRegression()
mlr_model.fit(X_train, y_train)

# Extract intercept and regression coefficients
intercept = mlr_model.intercept_
coefficients_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient (Beta)': mlr_model.coef_
}).sort_values(by='Coefficient (Beta)', ascending=False)

print(f"Model Intercept (Beta_0): {intercept:.4f}\n")
print("Feature Coefficients (Beta_i):")
coefficients_df

### Multiple Linear Regression Equation
The fitted linear equation is formulated as:
$$\hat{y} = \beta_0 + \beta_1(\text{age}) + \beta_2(\text{sex}) + \beta_3(\text{bmi}) + \beta_4(\text{children}) + \beta_5(\text{smoker}) + \beta_6(\text{region\_northwest}) + \beta_7(\text{region\_southeast}) + \beta_8(\text{region\_southwest})$$

## 7. Model Evaluation & Performance Metrics
Evaluate model performance on the unseen test dataset using standard regression metrics:
- **Mean Absolute Error (MAE)**: Average magnitude of absolute prediction errors.
- **Mean Squared Error (MSE)**: Average squared difference between actual and predicted charges.
- **Root Mean Squared Error (RMSE)**: Standard deviation of the prediction residuals.
- **Coefficient of Determination ($R^2$)**: Proportion of variance in insurance costs explained by the model.

In [ ]:
# Predict on training and testing partitions
y_train_pred = mlr_model.predict(X_train)
y_test_pred = mlr_model.predict(X_test)

# Compute metrics
mae = mean_absolute_error(y_test, y_test_pred)
mse = mean_squared_error(y_test, y_test_pred)
rmse = np.sqrt(mse)
r2_test = r2_score(y_test, y_test_pred)
r2_train = r2_score(y_train, y_train_pred)

print("================ MODEL EVALUATION METRICS ================")
print(f"Train R-squared (R2):         {r2_train:.4f}")
print(f"Test R-squared (R2):          {r2_test:.4f}")
print(f"Mean Absolute Error (MAE):    ${mae:.2f}")
print(f"Mean Squared Error (MSE):     {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): ${rmse:.2f}")
print("==========================================================")

In [ ]:
# Actual vs Predicted Charges Visualization
plt.figure(figsize=(9, 6))
plt.scatter(y_test, y_test_pred, color='#2980b9', alpha=0.6, edgecolors='k', s=50, label='Test Observations')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2.5, label='Perfect Prediction Line (y = x)')
plt.xlabel('Actual Charges ($)', fontsize=12)
plt.ylabel('Predicted Charges ($)', fontsize=12)
plt.title('Actual vs. Predicted Medical Insurance Charges (Test Set)', fontsize=14, fontweight='bold')
plt.legend()
plt.show()

In [ ]:
# Residual Analysis: Residual Distribution and Residuals vs Fitted Plot
residuals = y_test - y_test_pred

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(residuals, kde=True, color='#8e44ad', ax=ax[0])
ax[0].set_title('Residuals Distribution (Error Normality)', fontsize=13, fontweight='bold')
ax[0].set_xlabel('Residual (Actual - Predicted)')
ax[0].set_ylabel('Frequency')

sns.scatterplot(x=y_test_pred, y=residuals, color='#16a085', alpha=0.7, ax=ax[1])
ax[1].axhline(0, color='red', linestyle='--', lw=2)
ax[1].set_title('Residuals vs. Fitted Values (Homoscedasticity)', fontsize=13, fontweight='bold')
ax[1].set_xlabel('Fitted Charges ($)')
ax[1].set_ylabel('Residual ($)')
plt.tight_layout()
plt.show()

## 8. Sample Cost Prediction Tool
A practical helper function demonstrating real-world inference for newly profiled individuals.

In [ ]:
def predict_insurance_cost(age, sex, bmi, children, smoker, region):
    """
    Predict insurance charges based on input attributes.
    
    Parameters:
    - age: int (e.g., 25)
    - sex: 'male' or 'female'
    - bmi: float (e.g., 28.5)
    - children: int (e.g., 1)
    - smoker: 'yes' or 'no'
    - region: 'northeast', 'northwest', 'southeast', or 'southwest'
    """
    sex_enc = 1 if sex.lower() == 'female' else 0
    smoker_enc = 1 if smoker.lower() == 'yes' else 0
    reg_nw = 1 if region.lower() == 'northwest' else 0
    reg_se = 1 if region.lower() == 'southeast' else 0
    reg_sw = 1 if region.lower() == 'southwest' else 0
    
    sample_df = pd.DataFrame([{
        'age': age,
        'sex': sex_enc,
        'bmi': bmi,
        'children': children,
        'smoker': smoker_enc,
        'region_northwest': reg_nw,
        'region_southeast': reg_se,
        'region_southwest': reg_sw
    }])
    
    cost_pred = mlr_model.predict(sample_df)[0]
    return cost_pred

# Example Test Profiles
profile_1 = {'age': 25, 'sex': 'female', 'bmi': 22.5, 'children': 0, 'smoker': 'no', 'region': 'northeast'}
profile_2 = {'age': 45, 'sex': 'male', 'bmi': 33.2, 'children': 2, 'smoker': 'yes', 'region': 'southeast'}

cost_1 = predict_insurance_cost(**profile_1)
cost_2 = predict_insurance_cost(**profile_2)

print("Sample Inferences:")
print(f"Profile 1 (Healthy, Young Non-Smoker): Estimated Charge = ${cost_1:,.2f}")
print(f"Profile 2 (Middle-aged, Obese Smoker):    Estimated Charge = ${cost_2:,.2f}")

## 9. Key Findings & Conclusion
1. **Smoking Status**: The single largest predictor of healthcare charges. Smoking incurs an additional charge penalty exceeding **$23,600+**.
2. **Age & BMI**: Each additional year of age increases charges by ~$257, and each additional unit of BMI adds ~$339.
3. **Model Accuracy**: The Multiple Linear Regression model achieves an **$R^2$ score of ~78.4%**, demonstrating strong explanatory power for actuarial baseline modeling.